In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mahotas as mh
import imutils
import nd2
import cv2
import os
import pandas as pd
from scipy.signal import find_peaks
pwd = os.getcwd()

In [ ]:
IMAGE_SCALE = 0.5605 # pixel/micron
SCALE_BAR = 200 # micron
SCALE_BAR_MARGIN = 50 # pixels
SCALE_BAR_THICKNESS = 20 # pixels

CONVERT = True
CROP = True
PERIMETER = True

In [ ]:
# Get the folder address from the user
while True: 
    folder_address = input("Enter folder address: ")
    if os.path.isdir(folder_address):
        print("Folder path is correct.")
        break
    else:
        print("Folder path is not correct! Try again.")
        
# Create DataFrame to store perimeter and area data
data_columns = ['Folder', 'Image', 'Patient line', 'Sample ID', 'Recovery Day', 'Sample number', 'Organoid number', 'Perimeter', 'Area']
perimeter_data = pd.DataFrame(columns=data_columns)

In [ ]:
for (root,dirs,files) in os.walk(folder_address, topdown=True):
    if files: 
        ###########################################################
        ###########################################################
        ## CONVERT .nd2 files to .tif
        
        if CONVERT:
            all_imgs_names = [f for f in files \
                              if (os.path.splitext(f)[1] == '.nd2')]

            for img_num in range(len(all_imgs_names)):
                img_array = nd2.imread(os.path.join(root, all_imgs_names[img_num])) # read to numpy array
                image_name = all_imgs_names[img_num]
                cv2.imwrite(os.path.join(root, os.path.splitext(image_name)[0]+'.tif'), img_array)
        
        ###########################################################
        ###########################################################
        ## PROCESSING .tif images
        
        all_imgs_names = [f for f in files \
                          if (os.path.splitext(f)[1] == '.tif')]
        
        for img_num in range(len(all_imgs_names)):
            # Load the color image
            img_color = cv2.imread(os.path.join(root, all_imgs_names[img_num]),
                                   cv2.IMREAD_UNCHANGED)
            # Load the grayscale image
            bf_img = cv2.imread(os.path.join(root, all_imgs_names[img_num]), 
                                cv2.IMREAD_GRAYSCALE)

            bf_img = bf_img * int(255/np.max(bf_img))
            # plt.figure()
            # plt.imshow(bf_img, 'gray')
            # plt.show()

            ## OLD masking algorithm - start
#             histogram, bins = np.histogram(bf_img.ravel(), 256, [0,256])
#             w = 4
#             histogram = np.convolve(histogram, np.ones(w), 'valid') / w
#             # print(histogram)
#             # plt.figure()
#             # plt.plot(histogram)
#             # plt.show()


#             # for local maxima
#             maxima_indices = find_peaks(histogram, distance= 50)
#             bin_max = np.max(maxima_indices[0])
#             bin_min = np.min(maxima_indices[0])
#             bin_mean = np.mean([bin_min, bin_max])

#             ret, mask_img = cv2.threshold(bf_img, bin_mean, 255, cv2.THRESH_BINARY_INV)

#             plt.figure()
#             plt.imshow(mask_img, 'gray')
#             plt.show()
            ## OLD masking algorithm - end
    
            mask_img = cv2.adaptiveThreshold(bf_img,255,cv2.ADAPTIVE_THRESH_MEAN_C,\
                                              cv2.THRESH_BINARY_INV,699,3)
                
            labeled_img, n_img = mh.label(mask_img)
            labeled_img, n_img = mh.labeled.filter_labeled(labeled_img, 
                                                           remove_bordering=True, 
                                                           min_size=100000) # minimum size of PDOs - it helps to remove non-PDO objects
            # plt.figure()
            # plt.imshow(labeled_img, 'gray')
            # plt.show()
            nn = 1
            
            ###########################################################
            ###########################################################
            ## PROCESSING the PDOs found in each image
        
            for obj in np.unique(labeled_img):
                # if the label is zero, we are examining the 'background', so simply ignore it
                if obj == 0:
                    continue

                # otherwise, allocate memory for the label region and draw it on the mask
                mask = np.zeros(labeled_img.shape, dtype="uint8")
                mask[labeled_img == obj] = 255

                # detect contours in the green mask and grab the largest one
                cnts = cv2.findContours(mask.copy(), 
                                        cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
                cnts = imutils.grab_contours(cnts)
                c = max(cnts, key=cv2.contourArea)

                ###########################################################
                ###########################################################
                ## CROP the IMAGE
                
                # draw a circle enclosing the object
                ((x_obj, y_obj), r_obj) = cv2.minEnclosingCircle(c)
                cv2.circle(img_color, (int(x_obj), int(y_obj)), int(r_obj), 255, 2)
                # plt.figure()
                # plt.imshow(img_color, 'gray')
                # plt.show()

                CROP_W_SIZE = int(r_obj*2*1.25) 
                CROP_H_SIZE = int(r_obj*2*1.25) 

                # Calculate cropping parameters
                x = int(x_obj - CROP_H_SIZE/2)
                y = int(y_obj - CROP_W_SIZE/2)
                h = CROP_H_SIZE
                w = CROP_W_SIZE

                # Perform boundary checks
                x = max(0, x)
                y = max(0, y)
                h = min(h, bf_img.shape[0] - y)
                w = min(w, bf_img.shape[1] - x)

                # Crop the image
                bf_img_copy = bf_img.copy()                # first, make a copy of the original image
                img_cropped = bf_img_copy[y:y+h, x:x+w]    # second, crop the desired area   


                # Adjust SCALE_BAR_LENGTH based on the actual size of the cropped image
                scaled_bar_length = int (SCALE_BAR * IMAGE_SCALE) 
                scaled_bar_margin = int(SCALE_BAR_MARGIN * (w / CROP_W_SIZE))

                # Adjust start_point based on the actual size of the cropped image
                start_point = (w - scaled_bar_margin - scaled_bar_length, 
                               h - SCALE_BAR_MARGIN - SCALE_BAR_THICKNESS)

                # Draw the rectangle on the cropped image
                end_point = (w - scaled_bar_margin, 
                             h - SCALE_BAR_MARGIN)
                img_cropped = cv2.rectangle(img_cropped, start_point, end_point, 0, -1)

                # Display the result
                # plt.figure()
                # plt.imshow(img_cropped, 'gray')
                # plt.show()

                # Save the cropped image
                if CROP:
                    cv2.imwrite(os.path.join(root, 
                                             os.path.splitext(all_imgs_names[img_num])[0]+'_'+str(nn)+'.tif'), 
                                img_cropped)
            
                ###########################################################
                ###########################################################
                ## CALCULATE the PERIMETER
                
                if PERIMETER :
                    # Approximate the contour with a more detailed shape
                    epsilon = 0.0001 * cv2.arcLength(c, True)  # Adjust the epsilon value as needed
                    approx = cv2.approxPolyDP(c, epsilon, True)
                    scale_factor = 1 / IMAGE_SCALE  # Convert pixels/micron to microns/pixel

                    # Calculate perimeter of the approximated contour
                    perimeter_pixels = cv2.arcLength(approx, True)
                    perimeter_microns = perimeter_pixels * scale_factor
                    # print("Perimeter of object", obj, ":", perimeter_microns, "microns")

                    area = cv2.contourArea(c) * (1 / (IMAGE_SCALE ** 2))  # Convert pixels^2 to square microns

                    ###########################################################
                    ###########################################################
                    ## SAVE the RESULTS into the DATAFRAME

                    ## folder name : C18 - Patient 58 - day 21
                    folder_name = os.path.basename(root)
                    folder_name_split = folder_name.split(' - ')
                    sample_ID = folder_name_split[0]
                    patient_line = folder_name_split[1]
                    recovery_day = folder_name_split[2].split(' ')[1]
                    sample_number = all_imgs_names[img_num].split(' ')[0]
                    organoid_number = nn

                    # Add data to DataFrame
                    perimeter_data.loc[len(perimeter_data)] = {'Folder': folder_name,
                                                               'Image': all_imgs_names[img_num],
                                                               'Patient line': patient_line,
                                                               'Sample ID': sample_ID,
                                                               'Recovery Day': recovery_day,
                                                               'Sample number': sample_number, 
                                                               'Organoid number': organoid_number,
                                                               'Perimeter': perimeter_microns, 
                                                               'Area': area}


                    # Draw a line on the perimeter of the object
                    colored_img = cv2.cvtColor(bf_img, cv2.COLOR_BGR2RGB)
                    cv2.drawContours(colored_img, [approx], -1, (255, 0, 0), 2)  # Draw contour in blue color with thickness 2
                    # Crop the image
                    colored_img_cropped = colored_img[y:y+h, x:x+w]
                    colored_img_cropped = cv2.rectangle(colored_img_cropped, start_point, end_point, 0, -1)
                    # Display the new color image with the line drawn on the perimeter
                    # plt.figure()
                    # plt.imshow(colored_img)
                    # plt.title('Color Image with Perimeter')
                    # plt.show()


                    # Save the cropped image
                    cv2.imwrite(os.path.join(root, os.path.splitext(all_imgs_names[img_num])[0]+'_'+str(nn)+'perimeter.tif'),
                                colored_img_cropped)
                         
                nn += 1
                
###########################################################
###########################################################
## SAVE the perimeter data to Excel
if PERIMETER :
    excel_filename = "perimeter_data_sample.xlsx"
    # perimeter_data['Day'] = perimeter_data['Day'].astype(int)
    # perimeter_data.sort_values('Day', ascending=True, inplace=True)
    perimeter_data.to_excel(os.path.join(folder_address, excel_filename), index=False)
    print(f"Perimeter data saved to {excel_filename}")
    
print("Done !!")